# FOL Quiz Solver — NL Premises → Options → Z3
Flow đầy đủ: NL text → LLM sinh FOL (premises + options) → Z3 kiểm tra entailment → đáp án đúng.

In [43]:
# Microsoft Z3 trên PyPI là gói **z3-solver**. Gói tên **z3** (riêng) thường là stub → không có DeclareSort.
%pip uninstall -y z3
%pip install -q --upgrade z3-solver transformers accelerate

## Cell 1 — CompetitionSolver (Z3 engine)

In [44]:
import subprocess
# Xem z3 đang install từ đâu
result = subprocess.run(['pip', 'show', 'z3', 'z3-solver'], capture_output=True, text=True)
print(result.stdout)

# Uninstall z3 giả, install z3-solver thật
subprocess.run(['pip', 'uninstall', 'z3', '-y'], capture_output=True)
result2 = subprocess.run(['pip', 'install', 'z3-solver', '--force-reinstall'], 
                         capture_output=True, text=True)
print(result2.stdout[-500:])
print(result2.stderr[-300:])

Name: z3-solver
Version: 4.16.0.0
Summary: an efficient SMT solver library
Home-page: https://github.com/Z3Prover/z3
Author: The Z3 Theorem Prover Project
Author-email: 
License: MIT License
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 

  Using cached z3_solver-4.16.0.0-py3-none-manylinux_2_27_x86_64.whl.metadata (602 bytes)
Using cached z3_solver-4.16.0.0-py3-none-manylinux_2_27_x86_64.whl (31.7 MB)
  Attempting uninstall: z3-solver
    Found existing installation: z3-solver 4.16.0.0
    Uninstalling z3-solver-4.16.0.0:
      Successfully uninstalled z3-solver-4.16.0.0




In [45]:
import z3, subprocess

print("z3 file:", z3.__file__)  # xem path

# Uninstall z3 giả
r1 = subprocess.run(['pip', 'uninstall', 'z3', '-y'], capture_output=True, text=True)
print("uninstall z3:", r1.stdout, r1.stderr)

# Install z3-solver thật
r2 = subprocess.run(['pip', 'install', 'z3-solver', '--force-reinstall', '--quiet'], 
                    capture_output=True, text=True)
print("install z3-solver:", r2.stdout, r2.stderr)

z3 file: /usr/local/lib/python3.12/dist-packages/z3/__init__.py
uninstall z3:  WARNING: Skipping z3 as it is not installed.

install z3-solver:  


In [46]:
import z3
print(z3.get_version_string())  # phải ra "4.x.x.x"

4.16.0


In [47]:
from __future__ import annotations

import re

from z3 import (
    And,
    BoolRef,
    BoolSort,
    Const,
    DeclareSort,
    Exists,
    ExprRef,
    ForAll,
    Function,
    Implies,
    IntSort,
    IntVal,
    Not,
    Or,
    Solver,
    unsat,
)


class CompetitionSolver:
    def __init__(self):
        self.Object = DeclareSort("Object")
        self.predicates: dict = {}
        self.constants: dict = {}
        self.int_funcs: dict = {}

    def _term(self, name: str, env: dict) -> ExprRef:
        if name in env:
            return env[name]
        if name not in self.constants:
            self.constants[name] = Const(name, self.Object)
        return self.constants[name]

    def _get_pred(self, name: str, arity: int):
        key = (name.strip(), arity)
        if key not in self.predicates:
            if arity == 1:
                self.predicates[key] = Function(name, self.Object, BoolSort())
            elif arity == 2:
                self.predicates[key] = Function(name, self.Object, self.Object, BoolSort())
            else:
                raise ValueError(f"Chưa hỗ trợ arity={arity}")
        return self.predicates[key]

    def _get_int_unary(self, name: str):
        name = name.strip()
        if name not in self.int_funcs:
            self.int_funcs[name] = Function(name, self.Object, IntSort())
        return self.int_funcs[name]

    @staticmethod
    def _split_top(s: str, sep: str) -> list[str]:
        parts, buf, depth = [], [], 0
        for c in s:
            if c == "(": depth += 1
            elif c == ")": depth -= 1
            if c == sep and depth == 0:
                parts.append("".join(buf).strip())
                buf = []
            else:
                buf.append(c)
        parts.append("".join(buf).strip())
        return [p for p in parts if p]

    @staticmethod
    def _split_arrow_top(s: str) -> tuple[str, str | None]:
        depth = 0
        for i, c in enumerate(s):
            if c == "(": depth += 1
            elif c == ")": depth -= 1
            elif c == "→" and depth == 0:
                return s[:i].strip(), s[i + 1:].strip()
        return s.strip(), None

    @staticmethod
    def _strip_outer_parens(s: str) -> str:
        s = s.strip()
        while s.startswith("(") and s.endswith(")"):
            depth = 0
            balanced = True
            for j, c in enumerate(s):
                if c == "(": depth += 1
                elif c == ")":
                    depth -= 1
                    if depth == 0 and j < len(s) - 1:
                        balanced = False
                        break
            if balanced and depth == 0:
                s = s[1:-1].strip()
            else:
                break
        return s

    def parse_atom(self, atom_str: str, env: dict) -> BoolRef:
        atom_str = atom_str.strip()
        neg = atom_str.startswith("¬")
        if neg:
            atom_str = atom_str[1:].strip()
        m_cmp = re.match(
            r"^([a-zA-Z0-9_]+)\(\s*([a-zA-Z0-9_]+)\s*\)\s*(>=|<=|≥|≤|>|<|=)\s*(\d+)\s*$",
            atom_str,
        )
        if m_cmp:
            fname, arg, op, kstr = m_cmp.group(1), m_cmp.group(2), m_cmp.group(3), m_cmp.group(4)
            k = IntVal(int(kstr))
            t = self._get_int_unary(fname)(self._term(arg, env))
            expr = {">": t > k, "<": t < k, "=": t == k,
                    ">=": t >= k, "≥": t >= k, "<=": t <= k, "≤": t <= k}[op]
            return Not(expr) if neg else expr
        m = re.match(
            r"^([a-zA-Z0-9_]+)\(\s*([a-zA-Z0-9_]+)\s*(?:,\s*([a-zA-Z0-9_]+))?\s*\)\s*$",
            atom_str,
        )
        if not m:
            raise ValueError(f"Không parse được atom: {atom_str!r}")
        pred, a1, a2 = m.group(1), m.group(2), m.group(3)
        if a2 is None:
            expr = self._get_pred(pred, 1)(self._term(a1, env))
        else:
            expr = self._get_pred(pred, 2)(self._term(a1, env), self._term(a2, env))
        return Not(expr) if neg else expr

    def parse_complex_side(self, side_str: str, env: dict) -> BoolRef:
        s = self._strip_outer_parens(side_str.strip())
        # Tồn tại: ∃x, φ  hoặc  Exists(x, φ) — phải xử lý trước ∧/∨ để không tách nhầm phạm vi
        m_ex = re.match(r"^\s*(?:Exists\s*\(\s*|∃\s*)(\w+)\s*,\s*", s, re.I)
        if m_ex:
            vname = m_ex.group(1)
            rest = s[m_ex.end() :].strip()
            qv = Const(vname, self.Object)
            child_env = {**env, vname: qv}
            inner = self.parse_complex_side(rest, child_env)
            return Exists(qv, inner)
        if "∨" in s:
            parts = self._split_top(s, "∨")
            if len(parts) > 1:
                return Or(*[self.parse_complex_side(p, env) for p in parts])
        if "∧" in s:
            parts = self._split_top(s, "∧")
            if len(parts) > 1:
                return And(*[self.parse_complex_side(p, env) for p in parts])
        if s.startswith("¬"):
            return Not(self.parse_complex_side(s[1:].strip(), env))
        return self.parse_atom(s, env)

    def string_to_z3_expr(self, fol_string: str) -> BoolRef:
        s = fol_string.strip()
        qnames: list[str] = []
        while True:
            m = re.match(r"^\s*ForAll\(\s*(\w+)\s*,\s*", s, re.I)
            if not m:
                break
            qnames.append(m.group(1))
            s = s[m.end():].strip()
        for _ in qnames:
            if s.endswith(")"):
                s = s[:-1].strip()
        s = self._strip_outer_parens(s)
        env = {qn: Const(qn, self.Object) for qn in qnames}
        left, right = self._split_arrow_top(s)
        if right is None:
            final_expr = self.parse_complex_side(left, env)
        else:
            final_expr = Implies(
                self.parse_complex_side(left, env),
                self.parse_complex_side(right, env),
            )
        for qn in reversed(qnames):
            final_expr = ForAll(env[qn], final_expr)
        return final_expr

    def evaluate_quiz(self, premises_list: list[str], options_dict: dict[str, str]) -> str | None:
        base_solver = Solver()
        print("\n[Z3] Nạp premises...")
        for i, p_str in enumerate(premises_list, 1):
            try:
                z3_expr = self.string_to_z3_expr(p_str)
                base_solver.add(z3_expr)
                print(f"  P{i} OK: {p_str}")
            except Exception as e:
                print(f"  P{i} ERROR ({e}): {p_str}")
                raise

        print("\n[Z3] Kiểm tra options (phản chứng)...")
        correct_option = None
        for label in sorted(options_dict):
            opt_str = options_dict[label]
            try:
                test_solver = Solver()
                test_solver.add(base_solver.assertions())
                opt_z3 = self.string_to_z3_expr(opt_str)
                test_solver.add(Not(opt_z3))
                status = test_solver.check()
                print(f"  Option {label}: {status}  | {opt_str}")
                if status == unsat:
                    correct_option = label
                    print(f"    ✓ UNSAT → {label} là đáp án đúng!")
            except Exception as e:
                print(f"  Option {label} ERROR ({e}): {opt_str}")
        return correct_option

print("✓ CompetitionSolver sẵn sàng.")
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Đổi thành 7B / 14B nếu có RAM

print(f"Đang load {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✓ Model loaded on {model.device}")
import json, re

# ──────────────────────────────────────────────────────────────────────────────
# System prompt: buộc model trả về JSON có "premises" và "options"
# ──────────────────────────────────────────────────────────────────────────────
FOL_QUIZ_SYSTEM = """
You convert a natural-language multiple-choice quiz into FOL formulas for a Z3 solver.

OUTPUT: ONE JSON object only — no markdown fences, no explanation.
Keys:
  "premises": JSON array of strings — universal rules + ground facts.
  "options":  JSON object with keys "A", "B", "C", "D" — each value a FOL string.

=== NL → FOL KEYWORD MAP (Logic_Based_Educational_Queries style; copy EXACT spellings) ===
Map phrases in the quiz to these tokens; reuse the SAME spelling in premises and options.
- "active" / active student / active status          → active_status(...)
- "completed courses" / number of courses (count)   → completed_courses(...) ONLY inside comparisons, e.g. completed_courses(sarah) = 4 or ... ≥ 5
- "eligible" / advanced classes / can take advanced → eligible_advanced(...)
- NL rule: eligible students need / require approval  → ForAll(x, eligible_advanced(x) → requires_approval(x))
  (Use the predicate name **requires_approval** in this universal rule.)
- NL fact: has approval / has administrative approval / possesses approval → **has_approval(person)**
  (Use **has_approval** for ground facts and inside MC options; do NOT rename to has_administrative_approval or needs_*.)
- Person name Sarah                                  → constant **sarah** (lowercase)
- "insufficient completed courses" / fewer than 5   → compare with completed_courses(sarah) < 5 inside options when needed

Approval naming rule for THIS benchmark:
- **requires_approval(x)** = obligation side in a ForAll linked to eligibility.
- **has_approval(x)** = whether the individual actually has approval (facts + options).
Do not merge these two names; follow the example below.

=== SYNTAX (strict; parser rejects anything else) ===
1. Connectives — Unicode ONLY: ∧   ∨   ¬   →   (implication is U+2192, not ASCII ->).
2. Quantifiers: ForAll(x, formula) or Exists(x, formula) or Unicode `∃x, formula` — comma after the variable.
3. Boolean atoms: pred(term) or pred(t1, t2). Identifiers: [a-zA-Z0-9_]+ only.
4. Integer-valued unary functions ONLY inside comparisons: func(term) OP n
   OP ∈ {>=, <=, >, <, =, ≥, ≤} with integer literal n. Example: completed_courses(sarah) = 4
5. Forbidden inside formulas: ↔  ∀  English words  LaTeX  ASCII /\ ->  !=
   Existential is OK: `∃x, φ` (Unicode U+2203 + comma) or `Exists(x, φ)` — same comma style as ForAll.

=== JSON shape ===
"premises" MUST be a flat JSON array of strings (not nested arrays/objects).

=== GOLD ONE-SHOT (premises must match this list character-for-character for the same story) ===
NL question line:
Based on the requirements, which statement about Sarah is correct?
A. She can take advanced classes because she has approval
B. She cannot take advanced classes due to insufficient completed courses
C. She is eligible but lacks approval
D. Her active status alone qualifies her

Assume the usual stem: active + ≥5 courses → eligible; eligible → requires approval; Sarah active; 4 courses; has approval.

Expected JSON (premises order + symbols fixed):
{"premises":["ForAll(x, (active_status(x) ∧ completed_courses(x) ≥ 5) → eligible_advanced(x))","ForAll(x, eligible_advanced(x) → requires_approval(x))","active_status(sarah)","completed_courses(sarah) = 4","has_approval(sarah)"],"options":{"A":"eligible_advanced(sarah) ∧ has_approval(sarah)","B":"¬eligible_advanced(sarah) ∧ (completed_courses(sarah) < 5)","C":"eligible_advanced(sarah) ∧ ¬has_approval(sarah)","D":"active_status(sarah) → eligible_advanced(sarah)"}}
""".strip()


def build_user_prompt(nl_text: str) -> str:
    return (
        "Convert the following quiz into the JSON format described in the system message.\n"
        "Follow the NL→FOL keyword map: eligible_advanced, active_status, completed_courses in comparisons, "
        "requires_approval inside the ForAll rule for approval obligation, has_approval for facts and options, constant sarah.\n"
        "Use Unicode ∧ ∨ ¬ → and ForAll(x, ...). Option strings must mirror the gold one-shot style.\n\n"
        "QUIZ TEXT:\n"
        + nl_text.strip()
        + "\n\nOutput ONLY the JSON object."
    )


# ──────────────────────────────────────────────────────────────────────────────
# JSON repair: sửa lỗi phổ biến từ LLM
# ──────────────────────────────────────────────────────────────────────────────
def repair_and_parse_json(text: str) -> dict:
    # 1. Bóc ra chuỗi JSON nếu bị bọc trong markdown
    m = re.search(r"```(?:json)?\s*(.+?)\s*```", text, re.DOTALL | re.I)
    if m:
        text = m.group(1)

    # 2. Trích object JSON đầu tiên — đếm { } có nhận biết chuỗi "..." (tránh lệch vì FOL / nháy lồng)
    def _extract_first_json_object(s: str, start: int) -> tuple[str, int] | None:
        if start < 0 or start >= len(s) or s[start] != "{":
            return None
        depth = 0
        in_str = False
        esc = False
        i = start
        while i < len(s):
            c = s[i]
            if in_str:
                if esc:
                    esc = False
                elif c == "\\":
                    esc = True
                elif c == '"':
                    in_str = False
                i += 1
                continue
            if c == '"':
                in_str = True
                i += 1
                continue
            if c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    return s[start : i + 1], i + 1
            i += 1
        return None

    if "{" not in text:
        raise ValueError("Không tìm thấy '{' trong output model.")

    search_from = 0
    for _ in range(32):
        start = text.find("{", search_from)
        if start < 0:
            break
        got = _extract_first_json_object(text, start)
        if got is None:
            search_from = start + 1
            continue
        candidate, _end = got
        fixed = candidate.replace("\u201c", '"').replace("\u201d", '"')
        fixed = fixed.replace("\u2018", "'").replace("\u2019", "'")
        fixed = re.sub(r",\s*([}\]])", r"\1", fixed)
        fixed = re.sub(
            r'\{\s*"([^"]*)"\s*\}', lambda m: json.dumps(m.group(1)), fixed
        )
        try:
            return json.loads(fixed)
        except json.JSONDecodeError:
            search_from = start + 1
            continue
    raise ValueError(
        "Không trích được JSON hợp lệ: thiếu ngoặc đóng, chuỗi FOL làm hỏng JSON, hoặc output bị cắt (thử tăng max_new_tokens)."
    )


print("✓ Prompt + repair helpers đã sẵn sàng.")
## Cell 4 — Hàm `run_quiz`: NL → LLM → Z3 → Đáp án
def run_quiz(
    nl_text: str,
    *,
    max_new_tokens: int = 1024,
    temperature: float = 0.1,
) -> str | None:
    """
    Nhận đoạn text NL (stem + options A-D), trả về nhãn đáp án đúng ('A','B','C','D') hoặc None.
    """
    # ── Step 1: Gọi LLM ──────────────────────────────────────────────────────
    messages = [
        {"role": "system", "content": FOL_QUIZ_SYSTEM},
        {"role": "user",   "content": build_user_prompt(nl_text)},
    ]
    chat_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)

    gen_kwargs = {"max_new_tokens": max_new_tokens}
    if temperature > 0:
        gen_kwargs.update({"do_sample": True, "temperature": temperature})
    else:
        gen_kwargs["do_sample"] = False

    generated = model.generate(**inputs, **gen_kwargs)
    new_ids = generated[:, inputs.input_ids.shape[1]:]
    raw_output = tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0]

    print("=" * 60)
    print("[LLM RAW OUTPUT]")
    print(raw_output)
    print("=" * 60)

    # ── Step 2: Parse JSON ───────────────────────────────────────────────────
    data = repair_and_parse_json(raw_output)

    premises: list[str] = data.get("premises", [])
    options:  dict[str, str] = data.get("options", {})

    if not isinstance(premises, list) or not premises:
        raise TypeError('"premises" phải là list không rỗng.')
    if not isinstance(options, dict) or not options:
        raise TypeError('"options" phải là dict không rỗng.')

    # Đảm bảo mỗi premise là string thuần (không phải dict/list)
    cleaned_premises = []
    for p in premises:
        if isinstance(p, dict):
            # Lấy value đầu tiên nếu model lỡ wrap thêm
            p = next(iter(p.values()))
        cleaned_premises.append(str(p).strip())

    print("\n[PARSED PREMISES]")
    for i, s in enumerate(cleaned_premises, 1):
        print(f"  {i}. {s}")
    print("\n[PARSED OPTIONS]")
    for k in sorted(options):
        print(f"  {k}: {options[k]}")

    # ── Step 3: Z3 ──────────────────────────────────────────────────────────
    engine = CompetitionSolver()
    answer = engine.evaluate_quiz(cleaned_premises, options)

    print("\n" + "=" * 60)
    if answer:
        print(f"ĐÁP ÁN ĐÚNG (entailment): [ {answer} ]")
    else:
        print("Không tìm được đáp án duy nhất (không có option nào bị phản chứng).")
    print("=" * 60)
    return answer
print("✓ run_quiz đã sẵn sàng.")



<>:252: SyntaxWarning: invalid escape sequence '\ '
<>:252: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_16060/1221914577.py:252: SyntaxWarning: invalid escape sequence '\ '
  5. Forbidden inside formulas: ↔  ∀  English words  LaTeX  ASCII /\ ->  !=


✓ CompetitionSolver sẵn sàng.
Đang load Qwen/Qwen2.5-3B-Instruct...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✓ Model loaded on cuda:0
✓ Prompt + repair helpers đã sẵn sàng.
✓ run_quiz đã sẵn sàng.


In [48]:
## Cell 5 — Chạy thử với câu hỏi mẫu
SAMPLE_NL = """
Context:
Any active student who has completed at least 5 courses may register for advanced classes.
Any student registered for advanced classes must have administrative approval.
Sarah is an active student. Sarah has completed exactly 4 courses. Sarah has administrative approval.

Question: Which statement about Sarah is correct?
A. She can take advanced classes because she has approval.
B. She cannot take advanced classes because she has completed fewer than 5 courses.
C. She is eligible for advanced classes but lacks approval.
D. Her active status alone qualifies her for advanced classes.
"""

answer = run_quiz(SAMPLE_NL)

[LLM RAW OUTPUT]
{"premises":["ForAll(x, (active_status(x) ∧ completed_courses(x) ≥ 5) → eligible_advanced(x))","ForAll(x, eligible_advanced(x) → requires_approval(x))","active_status(sarah)","completed_courses(sarah) = 4","has_approval(sarah)"],"options":{"A":"eligible_advanced(sarah) ∧ has_approval(sarah)","B":"¬eligible_advanced(sarah) ∧ (completed_courses(sarah) < 5)","C":"eligible_advanced(sarah) ∧ ¬has_approval(sarah)","D":"active_status(sarah) → eligible_advanced(sarah)"}}

[PARSED PREMISES]
  1. ForAll(x, (active_status(x) ∧ completed_courses(x) ≥ 5) → eligible_advanced(x))
  2. ForAll(x, eligible_advanced(x) → requires_approval(x))
  3. active_status(sarah)
  4. completed_courses(sarah) = 4
  5. has_approval(sarah)

[PARSED OPTIONS]
  A: eligible_advanced(sarah) ∧ has_approval(sarah)
  B: ¬eligible_advanced(sarah) ∧ (completed_courses(sarah) < 5)
  C: eligible_advanced(sarah) ∧ ¬has_approval(sarah)
  D: active_status(sarah) → eligible_advanced(sarah)

[Z3] Nạp premises...
  P1

## Cell 5b — Option NL → **FOL** (output mong đợi) + Z3 entailment

Bạn chỉ điền **premises FOL** + **câu hỏi NL** + **list option tiếng Anh/Việt thô** (đúng thứ tự).

- **Output mong đợi:** LLM trả về từng dòng option đã **chuyển sang FOL** (hiển thị khối `[OUTPUT MONG ĐỢI — …]`).
- Sau đó ô tự gọi `evaluate_quiz` trên các FOL vừa sinh (nhãn A, B, … chỉ để log Z3).

**Điều kiện:** đã chạy ô có `CompetitionSolver`, **`model` / `tokenizer`**, và **`repair_and_parse_json`** (cùng ô Qwen + helpers).

In [50]:
# ── Sửa 3 khối: câu hỏi NL + premises FOL + list option NL thô ────────────────
# Output mong đợi: LLM chuyển mỗi option → một chuỗi FOL (trùng từ vựng premises), rồi Z3 entailment.

MY_QUESTION = """
Given the premises, which statement about John is correct?
"""

MY_PREMISES = [
       "ForAll(x, has_degree(x, MSc) → teach_undergrad(x))",
      "ForAll(x, ForAll(d, (higher(d, MSc) ∧ has_degree(x, d)) → teach_undergrad(x)))",
      "higher(PhD, MSc)",
      "higher(MSc, BSc)",
      "ForAll(a, ForAll(b, ForAll(c, (higher(a, b) ∧ higher(b, c)) → higher(a, c))))",
      "ForAll(x, department_head(x) → (∃d, has_degree(x, d) ∧ higher(d, BSc)))",
      "department_head(John) ∧ has_degree(John, PhD)"
]

MY_OPTIONS_TEXT = [
    "He can teach undergraduate courses",
    "He qualifies as department head but cannot teach",
    "He needs a Master's degree to teach undergraduates",
    "His PhD is insufficient for teaching",
]

OPTIONS_NL_TO_FOL_SYSTEM = """
You translate a multiple-choice quiz (question + NL options) into Boolean FOL formulas for Z3.
Use the PREMISES only for vocabulary (reuse EXACT predicate/function spellings).

STRICT FOL syntax:
- Connectives (Unicode only): ∧  ∨  ¬  →  (implication is U+2192, not ASCII ->).
- Quantifiers: ForAll(x, formula), Exists(x, formula), or Unicode `∃x, formula` — comma after the variable.
- Integer comparisons only as func(term) OP n with OP in >=, <=, >, <, =, ≥, ≤.

NL → symbol hints (when the story matches this pattern):
- "eligible" / advanced classes / can take advanced → eligible_advanced(...)
- "active" → active_status(...)
- "completed courses" / fewer than 5 / insufficient courses → completed_courses(...) in comparisons
- "has approval" / because she has approval / lacks approval → has_approval(...)  (options use this name even if premises mention requires_approval in a ForAll rule)

OUTPUT: ONE JSON object only — no markdown fences, no explanation.
Preferred shape (keys A,B,C,… in order):
{"options":{"A":"<fol>","B":"<fol>","C":"<fol>","D":"<fol>"}}
You may instead return {"options_fol":["<1st>","<2nd>",...]} in the same order as the input option list.

Each value must be ONE well-formed Boolean formula (ground atoms use ONLY constant names that actually appear in PREMISES for individuals in this case — e.g. John, not sarah).

CRITICAL — individuals / constants:
- NEVER copy the name sarah (or any name) from the ONE-SHOT below. The one-shot is ONLY an illustration of syntax + NL→FOL style.
- Read PREMISES and reuse the EXACT same constant spellings for people (e.g. has_degree(John, PhD) → use John in every option formula, never sarah).
- If several constants appear, prefer the person the question is about; still only symbols that appear in PREMISES.

Do NOT output the bare word False or True as the entire formula (parser expects predicates). Use a proper Boolean combination of atoms from PREMISES instead.

=== GOLD ONE-SHOT (style only — names sarah/eligible_advanced are specific to THAT story) ===
NL (question + options, verbatim lines):
Based on the requirements, which statement about Sarah is correct?
A. She can take advanced classes because she has approval
B. She cannot take advanced classes due to insufficient completed courses
C. She is eligible but lacks approval
D. Her active status alone qualifies her

Expected JSON (shape to mimic; YOUR task must use constants from YOUR premises, not sarah unless sarah appears there):
{"options":{"A":"eligible_advanced(sarah) ∧ has_approval(sarah)","B":"¬eligible_advanced(sarah) ∧ (completed_courses(sarah) < 5)","C":"eligible_advanced(sarah) ∧ ¬has_approval(sarah)","D":"active_status(sarah) → eligible_advanced(sarah)"}}
""".strip()


import json
import re


def _ground_constants_hint(premises: list[str]) -> str:
    """Trích các ký hiệu hằng (John, PhD, sarah, …) từ tham số trong ngoặc; bỏ biến lượng tử 1 ký tự x,d,y,z."""
    blob = "\n".join(premises)
    terms: set[str] = set()
    for a, b in re.findall(r"\(\s*([A-Za-z_]\w*)\s*,\s*([A-Za-z_]\w*)\s*\)", blob):
        for w in (a, b):
            if len(w) <= 1 or w in {"x", "d", "y", "z"}:
                continue
            terms.add(w)
    for pr in premises:
        if "ForAll" in pr:
            continue
        for m in re.finditer(r"\(\s*([A-Za-z_]\w*)\s*\)\s*$", pr.strip()):
            w = m.group(1)
            if len(w) > 1 and w not in {"x", "d", "y", "z"}:
                terms.add(w)
    return ", ".join(sorted(terms)) if terms else "(không trích được — hãy copy đúng chữ trong PREMISES)"


def _build_options_fol_user_prompt(premises, question, options_nl: list) -> str:
    n = len(options_nl)
    labels = [chr(ord("A") + i) for i in range(n)]
    hint = _ground_constants_hint(premises)
    return (
        "PREMISES (JSON list of FOL strings):\n"
        + json.dumps(premises, ensure_ascii=False)
        + "\n\nCONSTANTS / individuals detected in premises (use these spellings in options, do NOT invent sarah):\n"
        + hint
        + "\n\nQUESTION (NL):\n"
        + str(question).strip()
        + "\n\nOPTIONS (JSON array of NL strings, IN ORDER — index 0 = A, 1 = B, ...):\n"
        + json.dumps(options_nl, ensure_ascii=False)
        + "\n\nReturn ONLY JSON. Prefer: "
        + '{"options": {' + ", ".join(f'"{L}":"<FOL>"' for L in labels) + "}}"
        + " with exactly these keys and one formula string per key."
    )


def llm_options_nl_to_fol(
    premises: list[str],
    question: str,
    options_nl: list[str],
    *,
    max_new_tokens: int = 768,
    temperature: float = 0.1,
) -> list[str]:
    """Cần `model`, `tokenizer`, `repair_and_parse_json` từ ô load Qwen + helpers."""
    messages = [
        {"role": "system", "content": OPTIONS_NL_TO_FOL_SYSTEM},
        {"role": "user", "content": _build_options_fol_user_prompt(premises, question, options_nl)},
    ]
    chat_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)
    gen_kwargs: dict = {"max_new_tokens": max_new_tokens}
    if temperature and temperature > 0:
        gen_kwargs.update({"do_sample": True, "temperature": temperature})
    else:
        gen_kwargs["do_sample"] = False
    generated = model.generate(**inputs, **gen_kwargs)
    new_ids = generated[:, inputs.input_ids.shape[1] :]
    raw = tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0]
    print("=" * 60)
    print("[LLM — raw (JSON options → FOL)]")
    print(raw)
    print("=" * 60)
    data = repair_and_parse_json(raw)
    n = len(options_nl)
    labels_exp = [chr(ord("A") + i) for i in range(n)]

    if isinstance(data.get("options"), dict):
        d = data["options"]
        if all(L in d for L in labels_exp):
            out = [str(d[L]).strip() for L in labels_exp]
        else:
            keys = sorted(d.keys(), key=str)
            out = [str(d[k]).strip() for k in keys]
    elif isinstance(data.get("options_fol"), list):
        out = [str(x).strip() for x in data["options_fol"]]
    else:
        raise ValueError('Thiếu "options" (dict A/B/…) hoặc "options_fol" (list).')
    if len(out) != len(options_nl):
        raise ValueError(
            f"LLM trả {len(out)} công thức nhưng có {len(options_nl)} option — sửa prompt hoặc tăng max_new_tokens."
        )
    return out


# ── Chạy ─────────────────────────────────────────────────────────────────────

print("=" * 60)
print("[INPUT — CÂU HỎI]")
print(MY_QUESTION.strip() or "(để trống)")
print("=" * 60)

premises_clean = [str(p).strip() for p in MY_PREMISES if str(p).strip()]
opts_clean = [str(t).strip() for t in MY_OPTIONS_TEXT if str(t).strip()]

print("\n[PREMISES FOL]")
for i, p in enumerate(premises_clean, 1):
    print(f"  {i}. {p}")

print("\n[OPTIONS — NL thô]")
for i, t in enumerate(opts_clean, 1):
    print(f"  ({i}) {t}")

_missing = [n for n in ("model", "tokenizer", "repair_and_parse_json") if n not in globals()]
if _missing:
    raise RuntimeError(
        "Thiếu: "
        + ", ".join(_missing)
        + " — chạy ô load Qwen + helpers (cùng ô có CompetitionSolver) trước."
    )

fol_per_option = llm_options_nl_to_fol(premises_clean, MY_QUESTION, opts_clean)

print("\n" + "=" * 60)
print("[OUTPUT MONG ĐỢI — từng option đã chuyển sang FOL]")
labels = [chr(ord("A") + i) for i in range(len(fol_per_option))]
for lab, nl, fol in zip(labels, opts_clean, fol_per_option):
    print(f"  {lab})  NL:  {nl}")
    print(f"      FOL: {fol}")
print("=" * 60)

options_for_z3 = dict(zip(labels, fol_per_option))
eng = CompetitionSolver()
ans = eng.evaluate_quiz(premises_clean, options_for_z3)

print("\n" + "=" * 60)
if ans:
    idx = labels.index(ans)
    print(f"Z3 entailment: [{ans}]  |  NL: {opts_clean[idx]}")
else:
    print("Z3: không có option FOL nào được entail (xem SAT/UNSAT từng dòng phía trên).")
print("=" * 60)

[INPUT — CÂU HỎI]
Given the premises, which statement about John is correct?

[PREMISES FOL]
  1. ForAll(x, has_degree(x, MSc) → teach_undergrad(x))
  2. ForAll(x, ForAll(d, (higher(d, MSc) ∧ has_degree(x, d)) → teach_undergrad(x)))
  3. higher(PhD, MSc)
  4. higher(MSc, BSc)
  5. ForAll(a, ForAll(b, ForAll(c, (higher(a, b) ∧ higher(b, c)) → higher(a, c))))
  6. ForAll(x, department_head(x) → (∃d, has_degree(x, d) ∧ higher(d, BSc)))
  7. department_head(John) ∧ has_degree(John, PhD)

[OPTIONS — NL thô]
  (1) He can teach undergraduate courses
  (2) He qualifies as department head but cannot teach
  (3) He needs a Master's degree to teach undergraduates
  (4) His PhD is insufficient for teaching
[LLM — raw (JSON options → FOL)]
{"options": {"A":"teach_undergrad(John)", "B":"department_head(John) ∧ ¬teach_undergrad(John)", "C":"¬teach_undergrad(John) ∧ has_degree(John, MSc)", "D":"¬teach_undergrad(John) ∧ higher(has_degree(John, PhD), BSc) → teach_undergrad(John)rawl


ValueError: Không trích được JSON hợp lệ: thiếu ngoặc đóng, chuỗi FOL làm hỏng JSON, hoặc output bị cắt (thử tăng max_new_tokens).

## Cell 6 — Test offline (không cần model): kiểm tra Z3 với FOL hard-coded

In [ ]:
# Dùng cell này để debug Z3 mà không cần gọi LLM

test_premises = [
    "ForAll(x, (active_status(x) ∧ completed_courses(x) ≥ 5) → eligible_advanced(x))",
    "ForAll(x, eligible_advanced(x) → requires_approval(x))",
    "active_status(sarah)",
    "completed_courses(sarah) = 4",
    "has_approval(sarah)",
]

test_options = {
    "A": "eligible_advanced(sarah) ∧ has_approval(sarah)",
    "B": "¬eligible_advanced(sarah) ∧ (completed_courses(sarah) < 5)",
    "C": "eligible_advanced(sarah) ∧ ¬has_approval(sarah)",
    "D": "active_status(sarah) → eligible_advanced(sarah)",
}

engine = CompetitionSolver()
ans = engine.evaluate_quiz(test_premises, test_options)
print(f"\n→ Đáp án: {ans}")


[Z3] Nạp premises...
  P1 OK: ForAll(x, (active_status(x) ∧ completed_courses(x) ≥ 5) → eligible_advanced(x))
  P2 OK: ForAll(x, eligible_advanced(x) → requires_approval(x))
  P3 OK: active_status(sarah)
  P4 OK: completed_courses(sarah) = 4
  P5 OK: has_approval(sarah)

[Z3] Kiểm tra options (phản chứng)...
  Option A: sat  | eligible_advanced(sarah) ∧ has_approval(sarah)
  Option B: sat  | ¬eligible_advanced(sarah) ∧ (completed_courses(sarah) < 5)
  Option C: sat  | eligible_advanced(sarah) ∧ ¬has_approval(sarah)
  Option D: sat  | active_status(sarah) → eligible_advanced(sarah)

→ Đáp án: None
